In [1]:
"""
Main script for exaspim-to-template-to-CCF registration pipeline.

This module contains the main functions for performing CCF (Common Coordinate Framework)
registration of exaSPIM data to the Allen Mouse Brain Atlas.
"""

import json
import logging
import os
from datetime import datetime
from typing import List, Optional
import argparse

import ants
import numpy as np
import zarr
import dask.array as da
from numcodecs import blosc
import glob
import s3fs
from urllib.parse import urlparse
import shutil

from pathlib import Path
from aind_exaspim_ccf_reg.utils import (
    create_logger, 
    read_json_as_dict, 
    prepare_config_sample, 
    create_folder, 
    generate_processing,
    extract_dataset_id
)
from aind_exaspim_ccf_reg.configs import PathLike, RegSchema
from aind_exaspim_ccf_reg.preprocess import perc_normalization, check_orientation
from aind_exaspim_ccf_reg.plots import plot_reg, plot_antsimgs
from aind_data_schema.core.processing import DataProcess, ProcessName
from aind_exaspim_ccf_reg.register import RegistrationPipeline
from argschema import ArgSchemaParser

__version__ = "0.0.1"
code_url = "https://github.com/AllenNeuralDynamics/aind-exaspim-ccf-registration.git"

def load_zarr(
    image_path: PathLike, 
    logger: logging.Logger
) -> np.ndarray:
    """
    Load Zarr image.
    """
    image = zarr.open(image_path, mode="r")
    image = np.squeeze(np.squeeze(np.array(image), axis=0), axis=0)
    logger.info("----"*10)
    logger.info(f"Loading OMEZarr image from path: {image_path}")
    logger.info(f"image shape: {image.shape}")
    logger.info("----"*10)
        
    return image


def upload_alignment_data(
    s3_path: str,
    folder_to_upload: PathLike,
) -> str:
    """
    generate output meta data, processing.json
    Copies results to the destination bucket to make it available
    to scientists as soon as possible.

    Parameters
    ----------
    s3_path: str
        New dataset name where the data will
        be copied following the aind conventions
        e.g., s3://{bucket_path}/{new_dataset_name}

    folder_to_upload: PathLike
        Results folder path in Code Ocean

    Returns
    -------
    Tuple[str, str]
        The first position is the path where the dataset
        was moved. e.g., s3://{bucket_path}/{new_dataset_name}
        It includes the "s3://" prefix. 
        e.g., s3://{bucket_path}/{new_dataset_name}/{output_prediction}
    """

    #------------------------------------#
    # upload alignment results to s3
    #------------------------------------#
    # s3_path = f"s3://{bucket_path}/{new_dataset_name}"
    print(f"upload files to path {s3_path}")

    fs = s3fs.S3FileSystem()
    url = urlparse(s3_path)
    print(f"url: {url}")

    if url.scheme != "s3":
        raise NotImplementedError("Only s3 output_uri is supported, not {url.scheme}")
    
    print(f"uploading {folder_to_upload}")
    fs.put(
        folder_to_upload, url.netloc + url.path.rstrip("/") + "/", recursive=True, maxdepth=10
    ) 


def get_root_s3_prefix(s3_uri, levels_up=1):
    # Remove 's3://' and split path
    scheme, bucket_and_key = s3_uri.split('://', 1)
    bucket, *key_parts = bucket_and_key.split('/')
  
    # Go `levels_up` directories up from the current file path
    base_key = '/'.join(key_parts[:levels_up])
    return f's3://{bucket}/{base_key}/'


%matplotlib inline


## resample 721332 mask from 10um to 25um

In [2]:

brain = ants.image_read("/data/721332_whole_brain_mask/721332_10um_loaded_zarr_img.nii")
mask = ants.image_read("/data/721332_whole_brain_mask/721332_whole_brain_mask.nrrd")

print(brain)
print(mask)
brain = perc_normalization(brain)

plot_antsimgs(brain, 
  f"{outprefix}/exaspim_{dataset_id}",
  title=f"exaspim_{dataset_id}", 
  vmin=0, vmax=1.5)

plot_antsimgs(mask, 
  f"{outprefix}/exaspim_{dataset_id}_mask",
  title=f"exaspim_{dataset_id}_mask", 
  vmin=0, vmax=1)

In [ ]:
brain_25um = ants.image_read("/data/reg_721332_to_ccf_25um_v1.3/ccf_alignment/registration_metadata/721332_loaded_zarr_img.nii.gz")
print(brain_25um)

In [3]:
mask_25um = ants.resample_image(
    mask, brain_25um.spacing, interp_type=0
)
mask_25um

ANTsImage (ASL)
	 Pixel Type : float (float32)
	 Components : 1
	 Dimensions : (979, 393, 817)
	 Spacing    : (0.0203, 0.0203, 0.027)
	 Origin     : (0.0, 0.0, 0.0)
	 Direction  : [-0.  0. -1.  1. -0.  0.  0. -1.  0.]



In [7]:
# Get current size
print("Original size:", brain_25um.shape)  
# -> (980, 394, 817)
outprefix = "/results/"

# Define crop ranges: (start, end) for each axis
# If you want to remove 1 voxel from the end of X and Y
x_start, x_end = 0, brain_25um.shape[0]   # keep 0..978 (979 voxels)
y_start, y_end = 0, brain_25um.shape[1]   # keep 0..392 (393 voxels)
z_start, z_end = 0, brain_25um.shape[2]   # keep all (817 voxels)

cropped = ants.crop_indices(mask_25um,
                            (x_start, y_start, z_start),
                            (x_end, y_end, z_end))


Original size: (979, 393, 817)


In [10]:
outprefix = "/results/"
dataset_id = "721332"

plot_antsimgs(mask_25um, 
  f"{outprefix}/exaspim_{dataset_id}_mask_25um",
  title=f"exaspim_{dataset_id}_mask_25um", 
  vmin=0, vmax=1)

plot_antsimgs(cropped, 
  f"{outprefix}/exaspim_{dataset_id}_mask_25umcropped",
  title=f"exaspim_{dataset_id}_mask_25umcropped", 
  vmin=0, vmax=1)

In [18]:
mask_25um

ANTsImage (ASL)
	 Pixel Type : float (float32)
	 Components : 1
	 Dimensions : (980, 394, 817)
	 Spacing    : (0.0203, 0.0203, 0.027)
	 Origin     : (0.0, 0.0, 0.0)
	 Direction  : [ 0.  0. -1.  1.  0.  0.  0. -1.  0.]

In [11]:
ants.image_write(cropped, f"/data/721332_whole_brain_mask_25um.nii.gz")


## Load 997 mask for exaSPIM template

In [2]:
outprefix = "/results/"
ants_exaspim = ants.image_read("/data/exaSPIM_template_25um/exaspim_template_7sujects_nomask_25um_round6.nii.gz") # 25um
ccf = ants.image_read('../data/allen_mouse_ccf/average_template/average_template_25.nii.gz')

ccf = perc_normalization(ccf)
plot_antsimgs(ccf, 
              f"{outprefix}/ccf_template",
              title=f"ccf_template", 
              vmin=0, vmax=1.5)

# ants.image_write(ccf, f"{outprefix}load_ccf.nii.gz")
ants_exaspim.set_spacing( ccf.spacing )
ants_exaspim.set_origin( ccf.origin )
ants_exaspim.set_direction( ccf.direction )
print(f"Loaded ants_exaspim: {ants_exaspim}")
plot_antsimgs(ants_exaspim, 
              f"{outprefix}/exaspim_template",
              title=f"exaspim_template", 
              vmin=0, vmax=1.5)


Loaded ants_exaspim: ANTsImage (ASL)
	 Pixel Type : float (float32)
	 Components : 1
	 Dimensions : (648, 440, 576)
	 Spacing    : (0.025, 0.025, 0.025)
	 Origin     : (0.0, 0.0, 0.0)
	 Direction  : [-0.  0. -1.  1. -0.  0.  0. -1.  0.]



In [3]:
ccf

ANTsImage (ASL)
	 Pixel Type : float (float32)
	 Components : 1
	 Dimensions : (528, 320, 456)
	 Spacing    : (0.025, 0.025, 0.025)
	 Origin     : (0.0, 0.0, 0.0)
	 Direction  : [-0.  0. -1.  1. -0.  0.  0. -1.  0.]

In [4]:
exaspim_mask = ants.image_read("/data/root_annotation/root.nrrd")

In [5]:
exaspim_mask

ANTsImage (LPS)
	 Pixel Type : float (float32)
	 Components : 1
	 Dimensions : (456, 528, 320)
	 Spacing    : (0.025, 0.025, 0.025)
	 Origin     : (0.0, 13.175, 0.0)
	 Direction  : [-1.  0.  0.  0. -1.  0.  0.  0. -1.]

## Create mask for exaSPIM template

In [2]:

from skimage.measure import label
import scipy.ndimage as ni
from skimage.filters import threshold_li, threshold_otsu


class Masking:
    """
    Get a binary mask image from the given light sheet volume after
    thresholding. We compute the optimal threshold using Li thresholding
    """

    def __init__(self, ants_img, method):
        """Class constructor
        Parameters
        ----------
        ants_img: ANTsImage
            image from which mask will be computed.
        """
        self.ants_img = ants_img
        self.method = method
        

    def _getLargestCC(self, segmentation):
        """get the largest connected component"""
        labels = label(segmentation)
        assert labels.max() != 0  # assume at least 1 CC
        largestCC = labels == np.argmax(np.bincount(labels.flat)[1:]) + 1

        return largestCC

    def _get_threshold_li(self, arr_img: np.ndarray, method: str) -> float:
        """get the optimal threshold using Li thresholding"""
        start_time = datetime.now()
        print(start_time)
        
        if method == "li":  
            low_thresh = threshold_li(arr_img)
        if method == "otsu":
            low_thresh = threshold_otsu(arr_img)
        print(low_thresh)
        
        end_time = datetime.now()

        print(
            f"Find optimal threshold using Li thresholding, execution time:\
            {end_time - start_time} s -- low_thresh={low_thresh}"
        )
        return low_thresh

    def _cleanup_mask(self, arr_mask: np.ndarray) -> np.ndarray:
        """
        Morphological operations will be applied to clean up the mask by
        closing holes and eroding away small or weakly-connected areas.
        The following steps are applied:
            - Closing holes
            - Dilation with radius 1 voxel
            - Morphological closing
            - Retain largest component
        """
        # 3x3 structuring element with connectivity 2
        struct = ni.generate_binary_structure(3, 2)

        mask = ni.binary_fill_holes(arr_mask).astype(int)
        mask = ni.binary_dilation(mask, structure=struct).astype(int)
        mask = ni.binary_closing(mask).astype(int)
        mask = self._getLargestCC(mask)

        mask = ni.binary_dilation(mask, structure=struct, iterations=3).astype(int)
        arr_mask = ni.binary_fill_holes(mask).astype(int)

        return arr_mask

    def run(self) -> np.ndarray:
        """compute the mask"""
        arr_img = self.ants_img.numpy()
        
        arr_img = arr_img.astype(np.float32)

        # get optimal threshold using Li thresholding
        # https://scikit-image.org/docs/stable/auto_examples/developers/plot_threshold_li.html
        low_thresh = self._get_threshold_li(arr_img, self.method)

        # thresholding
        arr_mask = arr_img > low_thresh

        # clean up
        arr_mask = self._cleanup_mask(arr_mask)

        # convert numpy array to ants image
        ants_img_mask = ants.from_numpy(
            arr_mask.astype("float32"),
            spacing=self.ants_img.spacing,
            origin=self.ants_img.origin,
            direction=self.ants_img.direction,
        )

        return ants_img_mask 
    
    
def compute_mask(ants_img, method):
    print("Computing Mask")
    start_time = datetime.now()
    # Li thresholding + cleanup
    mask = Masking(ants_img, method)
    ants_img_mask = mask.run()
    end_time = datetime.now()

    print(
        f"Mask Complete, execution time: {end_time - start_time} s\
        -- image {ants_img_mask}"
    )
    return ants_img_mask
outprefix = "/results/"
# ants_exaspim = ants.image_read("/data/exaSPIM_template_25um/exaspim_template_7sujects_nomask_25um_round6.nii.gz") # 25um
# ccf = ants.image_read('../data/allen_mouse_ccf/average_template/average_template_25.nii.gz')

ants_exaspim = ants.image_read("/data/exaspim_template_7subjects_nomask_10um_round6_template_only/fixed_median.nii.gz") # 25um
ccf = ants.image_read('../data/allen_mouse_ccf/average_template/average_template_10.nii.gz')


ccf = perc_normalization(ccf)
plot_antsimgs(ccf, 
              f"{outprefix}/ccf_template",
              title=f"ccf_template", 
              vmin=0, vmax=1.5)

# ants.image_write(ccf, f"{outprefix}load_ccf.nii.gz")
ants_exaspim.set_spacing( ccf.spacing )
ants_exaspim.set_origin( ccf.origin )
ants_exaspim.set_direction( ccf.direction )
print(f"Loaded ants_exaspim: {ants_exaspim}")
plot_antsimgs(ants_exaspim, 
              f"{outprefix}/exaspim_template",
              title=f"exaspim_template", 
              vmin=0, vmax=1.5)




Loaded ants_exaspim: ANTsImage (ASL)
	 Pixel Type : float (float32)
	 Components : 1
	 Dimensions : (1620, 1100, 1440)
	 Spacing    : (0.01, 0.01, 0.01)
	 Origin     : (0.0, 0.0, 0.0)
	 Direction  : [-0.  0. -1.  1. -0.  0.  0. -1.  0.]



In [3]:
mask = compute_mask(ants_exaspim, "otsu")

plot_antsimgs(mask, 
              f"{outprefix}/exaspim_template_mask",
              title=f"exaspim_template_mask", 
              vmin=0, vmax=1)


Computing Mask
2025-08-20 23:26:09.724991
0.4092977
Find optimal threshold using Li thresholding, execution time:            0:00:54.865960 s -- low_thresh=0.4092977046966553
Mask Complete, execution time: 0:08:14.848209 s        -- image ANTsImage (ASL)
	 Pixel Type : float (float32)
	 Components : 1
	 Dimensions : (1620, 1100, 1440)
	 Spacing    : (0.01, 0.01, 0.01)
	 Origin     : (0.0, 0.0, 0.0)
	 Direction  : [-0.  0. -1.  1. -0.  0.  0. -1.  0.]



In [4]:
ants_exaspim

ANTsImage (ASL)
	 Pixel Type : float (float32)
	 Components : 1
	 Dimensions : (1620, 1100, 1440)
	 Spacing    : (0.01, 0.01, 0.01)
	 Origin     : (0.0, 0.0, 0.0)
	 Direction  : [-0.  0. -1.  1. -0.  0.  0. -1.  0.]

In [5]:
ants.image_write(mask, f"/data/exaSPIM_template_mask_10um_otsu.nii.gz")


In [8]:

ants_exaspim = ants_exaspim * mask
figpath = f"{outprefix}exaspim_template_masked"
plot_antsimgs(ants_exaspim, figpath, title=f"exaspim_template_masked")
